# Train a tiny resume parser on eight pages

This notebook fine-tunes granite-docling-258M, the smallest of the three students, on the six training pages in
`examples/`, then scores it on those pages and on two pages it has not seen. It runs the same code as the full
training, so every step of the pipeline can be watched and changed.

**Where to run it.** On Colab, choose *Runtime > Change runtime type > T4 GPU*; training takes a few minutes. On a
local machine, open it from the repository after `pip install -e ".[dev]"`; it uses a CUDA GPU or Apple MPS if one
is present and otherwise the CPU (about 6 minutes of training on an Apple M2 Max
(90 steps at about 4 seconds each)).

**What to expect.** Six pages teach the model the output format and let it memorise those pages; they do not teach
it to read new CVs. The full recipe in `docs/training.md` uses 1,393 pages.

Outputs go to `runs/example/` in the repository folder.

## 1. Setup

On Colab this clones the repository and installs the package (about 1 minute). Locally it only moves to the
repository root. `TINY_RESUME_PARSER_DATA` points the package at `examples/` and must be set before the package is
imported.

In [ ]:
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("tiny-resume-parser").exists():
        !git clone -q https://github.com/iamkucuk/tiny-resume-parser
    %cd tiny-resume-parser
    %pip install -q -e .
elif Path.cwd().name == "examples":
    os.chdir("..")

os.environ["TINY_RESUME_PARSER_DATA"] = "examples"
sys.path.insert(0, "src")  # an editable install is only seen by a restarted kernel

MODEL = "ibm-granite/granite-docling-258M"
OUT = Path("runs/example")
# The granite-docling recipe (configs/granite-docling-258m.toml): lr 1e-4, 12 image tiles, image noise on half the
# pages. Different here: 15 epochs and one optimizer step per page (--accum 1), because there are only six pages.
EPOCHS = 15
print(f"working in {Path.cwd()}, outputs to {OUT}")

## 2. One page and its label

Each line of `examples/train.jsonl` holds a page image path and the record the teacher model wrote for it. The
student learns to write the compact form below: the record as JSON with empty fields left out.

In [ ]:
from tiny_resume_parser.data import page_image, read_jsonl
from tiny_resume_parser.models import student_target

train_rows = read_jsonl("examples/train.jsonl")
eval_rows = read_jsonl("examples/eval.jsonl")
row = train_rows[0]
display(page_image(row).resize((496, 702)))
target = student_target(row["record"])
print(target[:1000], "...")
print(f"{len(train_rows)} training pages, {len(eval_rows)} unseen pages; this target is {len(target)} characters")

## 3. Train

This runs `tiny-resume-parser train` in the notebook process so the progress bar shows here. Before the first step
it prints a mask check: the number of image tokens (none of them learnt) and of answer tokens (the only ones in the
loss). On a T4 the run uses float32 because the T4 has no bf16. The batch is already one page; if it runs out of
memory, restart the runtime and close other notebooks.

In [ ]:
from tiny_resume_parser import train

args = f"--model {MODEL} --lr 1e-4 --max-patches 12 --augment 0.5 --epochs {EPOCHS} --accum 1"
train.main(f"{args} --train-file examples/train.jsonl --out {OUT}".split())

## 4. Training loss

The loss is the mean cross-entropy over the answer tokens. On six pages it falls from about 2 to below 0.01 (one run on
an Apple M2 Max): the model has memorised them.

In [ ]:
import json

import matplotlib.pyplot as plt

log = [e for e in json.loads((OUT / "log_history.json").read_text()) if "loss" in e]
plt.figure(figsize=(6, 3))
plt.plot([e["step"] for e in log], [e["loss"] for e in log])
plt.xlabel("optimizer step (one page)")
plt.ylabel("training loss")
plt.title("granite-docling-258M on six pages")
plt.show()
print(f"loss {log[0]['loss']:.2f} at step 1, {log[-1]['loss']:.3f} at step {log[-1]['step']}")

## 5. Score on seen and unseen pages

`evaluate` reads each page, compares the answer with the label field by field and writes one line per page to
`pages.jsonl` (docs/evaluation.md explains the score). In one run on an Apple M2 Max: field F1 0.89 on the six training
pages (all six answers valid JSON) and 0.00 on the two unseen pages (neither answer is valid JSON). On an Apple M2 Max a page takes 6 to 36 seconds; an answer that repeats
itself runs to the 2,500-token limit and takes longest.

In [ ]:
from tiny_resume_parser import evaluate

seen = evaluate.main(f"--model {OUT}/model --labels examples/train.jsonl --out {OUT}/eval_train".split())
unseen = evaluate.main(f"--model {OUT}/model --labels examples/eval.jsonl --out {OUT}/eval_unseen".split())
print(f"field F1 (strict): seen pages {seen['field_f1']:.2f}, unseen pages {unseen['field_f1']:.2f}")

## 6. Read an unseen answer

The answer below is for a page the model never saw. Its structure is right; the values show what six pages are not
enough for.

In [ ]:
page = json.loads((OUT / "eval_unseen" / "pages.jsonl").read_text().splitlines()[0])
display(page_image(eval_rows[0]).resize((496, 702)))
print(page["status"], "\n", page["answer"][:1500])

## 7. Repair the broken answers

An unparsed answer scores 0 even when its first fields are right. `rescore --repair` cuts a broken answer at the
point where the JSON breaks and keeps the complete values before it, as `tiny-resume-parser parse` does for real
use. In the same run: strict F1 0.06 and content F1 0.19 on the two unseen pages. The full models in the README were
trained on 1,393 pages and score 0.63 to 0.75 lenient F1 on 204 test pages.

In [ ]:
from tiny_resume_parser import rescore

rescore.main(f"--repair --labels examples/eval.jsonl {OUT}/eval_unseen".split())

## 8. Try the model on any page

Pick one of the eight example pages, or upload a page image or PDF (the first page is read), and press **Read page**.
The answer is repaired as `tiny-resume-parser parse` does. For an example page, the label is compared value by
value with the lenient rule (case, spaces and punctuation ignored): green values are in the label, red values are
not, and the list on the right shows label values the model left out. The cell also reads the first unseen page once,
so a saved copy of the notebook shows a result.

In [ ]:
import html
import io
import tempfile
from collections import Counter

import ipywidgets as widgets
from IPython.display import display

from tiny_resume_parser.generate import generate_pages, load_model
from tiny_resume_parser.metrics import lenient, pairs
from tiny_resume_parser.metrics import parse as parse_answer
from tiny_resume_parser.parse import page_images

model, processor, device = load_model(str(OUT / "model"))
choices = {f"{r['cv_id']} (unseen)": r for r in eval_rows} | {f"{r['cv_id']} (trained on)": r for r in train_rows}
pick = widgets.Dropdown(options=list(choices), description="Page", layout=widgets.Layout(width="420px"))
upload = widgets.FileUpload(accept=".png,.jpg,.jpeg,.pdf", multiple=False, description="Upload")
max_tokens = widgets.IntSlider(value=1500, min=200, max=3000, step=100, description="Max tokens")
button = widgets.Button(description="Read page", button_style="primary")
out = widgets.Output()


def uploaded_image():
    """The uploaded file's first page, or None (ipywidgets 7 stores uploads in a dict, 8 in a tuple)."""
    v = upload.value
    if not v:
        return None
    f = v[0] if isinstance(v, (tuple, list)) else next(iter(v.values()))
    name = f.get("name") or f["metadata"]["name"]
    with tempfile.NamedTemporaryFile(suffix=Path(name).suffix) as tmp:
        tmp.write(bytes(f["content"]))
        tmp.flush()
        return page_images(Path(tmp.name))[0]


def lenient_pair(pair):
    return next(iter(lenient(Counter({pair: 1}))))


def value_list(items, colour):
    rows = "".join(
        f"<li style='color:{colour}'>{html.escape(path)}: {html.escape(value)}</li>" for path, value in items
    )
    return f"<ul style='margin:0;padding-left:18px'>{rows}</ul>"


def read_page(_=None):
    out.clear_output()
    with out:
        image = uploaded_image()
        label = None if image is not None else choices[pick.value]["record"]
        image = image if image is not None else page_image(choices[pick.value])
        answer, n_tokens = generate_pages(model, processor, [image], 12, max_tokens.value)[0]
        record = parse_answer(answer, fix=True)
        png = io.BytesIO()
        image.resize((400, int(400 * image.height / image.width))).save(png, "PNG")
        right = f"<b>{n_tokens} tokens</b><pre style='max-height:560px;overflow:auto;font-size:11px'>"
        right += html.escape(json.dumps(record, ensure_ascii=False, indent=1)) + "</pre>"
        if label is not None:
            pred, gold = pairs(record), pairs(label)
            p, g = lenient(pred), lenient(gold)
            hit = sum((p & g).values())
            f1 = 2 * hit / max(sum(p.values()) + sum(g.values()), 1)
            right = f"<b>lenient field F1 {f1:.2f}: {hit} of {sum(g.values())} label values found</b><br>" + right
            shown = {lenient_pair(k): k for k in pred}  # lenient form -> the value as the model wrote it
            found = [shown[k] for k in p & g]
            wrong = [shown[k] for k in p - g]
            missed = [k for k in gold if lenient_pair(k) not in p]
            right += "<b>Found</b>" + value_list(found[:15], "green") + "<b>Not in the label</b>"
            right += value_list(wrong[:15], "crimson") + "<b>Left out</b>" + value_list(missed[:15], "gray")
        display(widgets.HBox([widgets.Image(value=png.getvalue(), format="png"), widgets.HTML(right)]))


button.on_click(read_page)
display(widgets.VBox([widgets.HBox([pick, upload]), widgets.HBox([max_tokens, button]), out]))
read_page()

## 9. Release the GPU (Colab)

Colab GPU time is limited; this ends the session. Copy anything from `runs/example/` first.

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import runtime

    runtime.unassign()